# Superstore Sales Dashboard
**Business Intelligence Analysis | Python + Plotly**

Analyzing 4 years of retail sales data (9,994 orders, 2014–2017) across revenue, profit, customer segments, and discount impact.

**Libraries used:** pandas, numpy, plotly, zipfile

**Output:** Interactive HTML dashboard (`superstore_dashboard.html`)

In [ ]:
import zipfile
import os
import pandas as pd

ZIP_PATH  = "./Sample - Superstore.csv.zip"
WORK_DIR  = "./"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    file_list = zf.namelist()
    print(f"Files inside zip: {file_list}")
    zf.extractall(WORK_DIR)

csv_filename = None
for name in file_list:
    if name.lower().endswith(".csv"):
        csv_filename = name
        break

if csv_filename is None:
    raise FileNotFoundError(f"No CSV found inside zip. Contents: {file_list}")

print(f"Reading CSV: {csv_filename}")

csv_path = os.path.join(WORK_DIR, csv_filename)
try:
    df = pd.read_csv(csv_path, encoding="utf-8")
    print("Encoding: utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, encoding="latin-1")
    print("Encoding: latin-1 (fallback)")

COLUMN_MAP = {
    "Order ID":       "order_id",
    "Order Date":     "order_date",
    "Ship Date":      "ship_date",
    "Ship Mode":      "ship_mode",
    "Customer ID":    "customer_id",
    "Customer Name":  "customer_name",
    "Segment":        "segment",
    "Country":        "country",
    "City":           "city",
    "State":          "state",
    "Region":         "region",
    "Product ID":     "product_id",
    "Category":       "category",
    "Sub-Category":   "sub_category",
    "Product Name":   "product_name",
    "Sales":          "sales",
    "Quantity":       "quantity",
    "Discount":       "discount",
    "Profit":         "profit",
}
df = df.rename(columns=COLUMN_MAP)

df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"]  = pd.to_datetime(df["ship_date"])

df["shipping_days"]     = (df["ship_date"] - df["order_date"]).dt.days
df["profit_margin_pct"] = (df["profit"] / df["sales"] * 100).round(2)
df["month_year"]        = df["order_date"].dt.to_period("M")

print(f"\n✅ Data shape: {df.shape}")
print(f"   Date range: {df['order_date'].min().date()} → {df['order_date'].max().date()}")
print(f"   Unique customers: {df['customer_id'].nunique():,}")
print(f"   Total revenue:   ${df['sales'].sum():>12,.0f}")
print(f"   Total profit:    ${df['profit'].sum():>12,.0f}")

In [ ]:
import pandas as pd
import numpy as np

# ── Configuration ──────────────────────────────────────────────────────────────
SNAPSHOT_DATE    = pd.Timestamp("2017-12-31")
GROWTH_TARGET    = 0.10          # 10% YoY growth
BASELINE_BUFFER  = 0.90          # 2014 baseline = overall avg * 0.90
PARETO_THRESHOLD = 80.0
DISCOUNT_LABELS  = ["0%", "1-10%", "11-20%", "21-30%", "31%+"]
DISCOUNT_BINS    = [-0.001, 0.0, 0.10, 0.20, 0.30, 1.0]

# ── Top-line KPIs ─────────────────────────────────────────────────────────────
total_revenue      = df["sales"].sum()
total_profit       = df["profit"].sum()
overall_margin_pct = (total_profit / total_revenue * 100)
total_orders       = len(df)
avg_order_value    = df["sales"].mean()
avg_ship_days      = df["shipping_days"].mean()

best_region   = df.groupby("region")["profit"].sum().idxmax()
best_category = (df.groupby("category")["profit"].sum()
                   / df.groupby("category")["sales"].sum() * 100).idxmax()
best_month    = (df.groupby("month_year")["sales"].sum().idxmax())

# ── Region & Category summaries ───────────────────────────────────────────────
region_summary = (df.groupby("region")
                    .agg(total_sales=("sales","sum"),
                         total_profit=("profit","sum"),
                         order_count=("order_id","count"))
                    .reset_index())
region_summary["margin_pct"] = (region_summary["total_profit"]
                                 / region_summary["total_sales"] * 100).round(2)

category_summary = (df.groupby("category")
                      .agg(total_sales=("sales","sum"),
                           total_profit=("profit","sum"),
                           order_count=("order_id","count"))
                      .reset_index())
category_summary["margin_pct"] = (category_summary["total_profit"]
                                   / category_summary["total_sales"] * 100).round(2)

# ── Pareto (sub-category) ─────────────────────────────────────────────────────
sub_cat_sales = (df.groupby("sub_category")["sales"].sum()
                   .sort_values(ascending=False)
                   .reset_index())
sub_cat_sales.columns = ["sub_category", "sales"]
sub_cat_sales["cumulative_pct"] = (sub_cat_sales["sales"].cumsum()
                                    / sub_cat_sales["sales"].sum() * 100).round(2)
sub_cat_sales["in_top_80"]      = sub_cat_sales["cumulative_pct"] <= PARETO_THRESHOLD

# ── Monthly Sales ─────────────────────────────────────────────────────────────
monthly_sales = (df.groupby("month_year")["sales"].sum()
                   .reset_index()
                   .sort_values("month_year"))
monthly_sales["month_year"] = monthly_sales["month_year"].astype(str)
monthly_sales.columns = ["month_year", "total_sales"]
monthly_sales["rolling_3m"] = monthly_sales["total_sales"].rolling(3, min_periods=1).mean().round(2)

# ── RFM Analysis ──────────────────────────────────────────────────────────────
rfm = (df.groupby("customer_id")
         .agg(last_purchase=("order_date","max"),
              frequency=("order_id","count"),
              monetary=("sales","sum"))
         .reset_index())
rfm["recency"] = (SNAPSHOT_DATE - rfm["last_purchase"]).dt.days

rfm["recency_score"]  = pd.qcut(rfm["recency"],  q=4, labels=[4, 3, 2, 1]).astype(int)
rfm["freq_score"]     = pd.qcut(rfm["frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm["monetary_score"] = pd.qcut(rfm["monetary"].rank(method="first"),  q=4, labels=[1, 2, 3, 4]).astype(int)

def rfm_segment(row):
    r, f, m = row["recency_score"], row["freq_score"], row["monetary_score"]
    if r == 4 and f == 4 and m == 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r == 1 and f <= 2:
        return "Lost"
    else:
        return "Potential"

rfm["rfm_segment"] = rfm.apply(rfm_segment, axis=1)

rfm_summary = (rfm.groupby("rfm_segment")
                   .agg(customer_count=("customer_id","count"),
                        avg_monetary=("monetary","mean"))
                   .reset_index()
                   .sort_values("customer_count", ascending=False))

# ── Discount Elasticity ───────────────────────────────────────────────────────
df["discount_bin"] = pd.cut(df["discount"], bins=DISCOUNT_BINS, labels=DISCOUNT_LABELS)
discount_analysis = (df.groupby("discount_bin", observed=True)
                       .agg(avg_margin_pct=("profit_margin_pct","mean"),
                            avg_sales=("sales","mean"),
                            order_count=("order_id","count"))
                       .reset_index())
discount_analysis["discount_bin"] = discount_analysis["discount_bin"].astype(str)

# ── Cohort Retention ──────────────────────────────────────────────────────────
work = df[["customer_id","order_date","month_year"]].copy()
work["order_month"] = work["order_date"].dt.to_period("M")
first_purchase = work.groupby("customer_id")["order_month"].min().rename("cohort_month")
work = work.join(first_purchase, on="customer_id")
work["cohort_index"] = (work["order_month"] - work["cohort_month"]).apply(lambda x: x.n)

cohort_pivot = (work.groupby(["cohort_month","cohort_index"])["customer_id"]
                    .nunique()
                    .unstack(fill_value=0))
cohort_sizes      = cohort_pivot[0]
MAX_COHORT_INDEX  = min(12, cohort_pivot.columns.max())
cohort_pivot      = cohort_pivot[[c for c in range(MAX_COHORT_INDEX + 1)
                                   if c in cohort_pivot.columns]]
retention_pct     = cohort_pivot.divide(cohort_sizes, axis=0).multiply(100).round(1)
retention_pct.index = retention_pct.index.astype(str)

# ── Target Gap ────────────────────────────────────────────────────────────────
monthly = df.copy()
monthly["ym"] = monthly["order_date"].dt.to_period("M")
monthly_rev = monthly.groupby("ym")["sales"].sum().reset_index()
monthly_rev.columns = ["ym","actual_sales"]
monthly_rev["year"]  = monthly_rev["ym"].dt.year
monthly_rev["month"] = monthly_rev["ym"].dt.month

pivot = monthly_rev.pivot_table(index="month", columns="year",
                                 values="actual_sales", aggfunc="sum")
overall_monthly_avg = monthly_rev.groupby("month")["actual_sales"].mean()

target_rows = []
for _, row in monthly_rev.iterrows():
    yr, mo = row["year"], row["month"]
    if yr > monthly_rev["year"].min():
        prev = pivot.get(yr - 1)
        target = prev[mo] * (1 + GROWTH_TARGET) if prev is not None and mo in prev.index else overall_monthly_avg[mo] * BASELINE_BUFFER
    else:
        target = overall_monthly_avg[mo] * BASELINE_BUFFER
    target_rows.append({"month_year": str(row["ym"]),
                         "actual_sales": row["actual_sales"],
                         "target_sales": round(target, 2)})

monthly_target_df = pd.DataFrame(target_rows).sort_values("month_year")
monthly_target_df["gap"]            = monthly_target_df["actual_sales"] - monthly_target_df["target_sales"]
monthly_target_df["hit"]            = monthly_target_df["gap"] >= 0
monthly_target_df["achievement_pct"] = (monthly_target_df["actual_sales"]
                                         / monthly_target_df["target_sales"] * 100).round(1)

# ── Print Summary ─────────────────────────────────────────────────────────────
print(f"{'═'*60}")
print(f"  COMPUTED KPI METRICS")
print(f"{'═'*60}")
print(f"  Total Revenue:       ${total_revenue:>12,.0f}")
print(f"  Total Profit:        ${total_profit:>12,.0f}")
print(f"  Overall Margin %:    {overall_margin_pct:>11.2f}%")
print(f"  Total Orders:        {total_orders:>12,}")
print(f"  Avg Order Value:     ${avg_order_value:>12,.2f}")
print(f"  Avg Shipping Days:   {avg_ship_days:>11.2f}")
print(f"  Best Region:         {best_region}")
print(f"  Best Category:       {best_category}")
print(f"\n  Region Summary:\n{region_summary.to_string(index=False)}")
print(f"\n  Category Summary:\n{category_summary.to_string(index=False)}")
print(f"\n  RFM Segments:\n{rfm_summary.to_string(index=False)}")
print(f"\n  Discount Analysis:\n{discount_analysis.to_string(index=False)}")
print(f"\n  Target Hit Rate: {monthly_target_df['hit'].sum()}/{len(monthly_target_df)} months")
print(f"  Avg Achievement: {monthly_target_df['achievement_pct'].mean():.1f}%")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# ── Color palette ─────────────────────────────────────────────────────────────
C_BLUE    = "#3498db"
C_GREEN   = "#27ae60"
C_ORANGE  = "#f39c12"
C_RED     = "#e74c3c"
C_PURPLE  = "#8e44ad"
C_TEAL    = "#16a085"
C_DARK    = "#2c3e50"
C_GREY    = "#95a5a6"
TEMPLATE  = "plotly_white"

# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 — KPI Cards
# ─────────────────────────────────────────────────────────────────────────────
fig_kpi = make_subplots(
    rows=1, cols=6,
    specs=[[{"type": "indicator"}] * 6],
    subplot_titles=["Revenue", "Profit", "Margin %", "Orders", "Avg Order", "Ship Days"]
)

kpi_items = [
    (total_revenue,       "Total Revenue",    "$,.0f",   C_BLUE,   None),
    (total_profit,        "Total Profit",     "$,.0f",   C_GREEN,  None),
    (overall_margin_pct,  "Margin %",         ".2f",     C_TEAL,   None),
    (total_orders,        "Total Orders",     ",d",      C_PURPLE, None),
    (avg_order_value,     "Avg Order Value",  "$,.2f",   C_ORANGE, None),
    (avg_ship_days,       "Avg Ship Days",    ".1f",     C_GREY,   None),
]

for i, (val, title, fmt, color, delta) in enumerate(kpi_items, 1):
    if fmt.startswith("$"):
        vfmt = fmt[1:]
        prefix = "$"
    else:
        prefix = ""
        vfmt = fmt

    suffix = " days" if "Ship" in title else ("%" if "Margin" in title else "")

    fig_kpi.add_trace(
        go.Indicator(
            mode="number",
            value=val,
            number={"prefix": prefix, "valueformat": vfmt,
                    "suffix": suffix,
                    "font": {"size": 28, "color": color}},
            title={"text": title, "font": {"size": 13, "color": C_DARK}},
        ),
        row=1, col=i
    )

fig_kpi.update_layout(
    template=TEMPLATE,
    height=200,
    title_text="Executive KPI Summary",
    title_font_size=16,
    margin=dict(t=50, b=10, l=10, r=10),
)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 — Pareto Analysis
# ─────────────────────────────────────────────────────────────────────────────
pareto_df = sub_cat_sales.copy()
bar_colors = [C_BLUE if in80 else C_GREY for in80 in pareto_df["in_top_80"]]

fig_pareto = make_subplots(specs=[[{"secondary_y": True}]])

fig_pareto.add_trace(
    go.Bar(
        x=pareto_df["sub_category"],
        y=pareto_df["sales"],
        marker_color=bar_colors,
        name="Sales ($)",
        customdata=np.stack([pareto_df["cumulative_pct"]], axis=-1),
        hovertemplate="<b>%{x}</b><br>Sales: $%{y:,.0f}<br>Cumulative: %{customdata[0]:.1f}%<extra></extra>",
    ),
    secondary_y=False,
)

fig_pareto.add_trace(
    go.Scatter(
        x=pareto_df["sub_category"],
        y=pareto_df["cumulative_pct"],
        mode="lines+markers",
        name="Cumulative %",
        line=dict(color=C_RED, width=2.5),
        marker=dict(size=6),
        hovertemplate="<b>%{x}</b><br>Cumulative %: %{y:.1f}%<extra></extra>",
    ),
    secondary_y=True,
)

fig_pareto.add_hline(y=80, line_dash="dash", line_color=C_ORANGE,
                      annotation_text="80% Threshold", secondary_y=True)

fig_pareto.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="Pareto Analysis — 80/20 Rule (Sub-Category Revenue)",
    title_font_size=15,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80, b=60, l=60, r=60),
)
fig_pareto.update_yaxes(title_text="Sales ($)", secondary_y=False)
fig_pareto.update_yaxes(title_text="Cumulative Revenue %", secondary_y=True, range=[0, 110])

# ─────────────────────────────────────────────────────────────────────────────
# CHART 3 — Revenue Donut by Category
# ─────────────────────────────────────────────────────────────────────────────
fig_donut = px.pie(
    category_summary,
    names="category",
    values="total_sales",
    hole=0.4,
    color_discrete_sequence=[C_BLUE, C_GREEN, C_ORANGE],
    title="Revenue Share by Category",
)
fig_donut.update_traces(
    textposition="outside",
    textinfo="percent+label",
    hovertemplate="<b>%{label}</b><br>Revenue: $%{value:,.0f}<br>Share: %{percent}<extra></extra>",
)
fig_donut.update_layout(
    template=TEMPLATE,
    height=450,
    title_font_size=15,
    margin=dict(t=80, b=20, l=20, r=20),
    showlegend=True,
)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 4 — Profit Margin by Category (Horizontal Bar)
# ─────────────────────────────────────────────────────────────────────────────
cat_sorted = category_summary.sort_values("margin_pct")
margin_colors = [C_GREEN if m >= 0 else C_RED for m in cat_sorted["margin_pct"]]

fig_margin = go.Figure(go.Bar(
    x=cat_sorted["margin_pct"],
    y=cat_sorted["category"],
    orientation="h",
    marker_color=margin_colors,
    text=[f"{m:.2f}%" for m in cat_sorted["margin_pct"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Margin: %{x:.2f}%<extra></extra>",
))
fig_margin.add_vline(x=0, line_width=1, line_color=C_DARK)
fig_margin.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="Profit Margin % by Category",
    title_font_size=15,
    xaxis_title="Profit Margin (%)",
    yaxis_title="Category",
    margin=dict(t=60, b=40, l=120, r=80),
)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 5 — Regional Performance (Grouped Bar)
# ─────────────────────────────────────────────────────────────────────────────
fig_regional = go.Figure()
fig_regional.add_trace(go.Bar(
    name="Total Sales",
    x=region_summary["region"],
    y=region_summary["total_sales"],
    marker_color=C_BLUE,
    hovertemplate="<b>%{x}</b><br>Sales: $%{y:,.0f}<extra></extra>",
))
fig_regional.add_trace(go.Bar(
    name="Total Profit",
    x=region_summary["region"],
    y=region_summary["total_profit"],
    marker_color=C_GREEN,
    hovertemplate="<b>%{x}</b><br>Profit: $%{y:,.0f}<extra></extra>",
))
fig_regional.update_layout(
    template=TEMPLATE,
    height=450,
    barmode="group",
    title_text="Regional Sales vs Profit",
    title_font_size=15,
    xaxis_title="Region",
    yaxis_title="Amount ($)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80, b=40, l=60, r=20),
)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 6 — Monthly Revenue Trend
# ─────────────────────────────────────────────────────────────────────────────
fig_trend = go.Figure()
fig_trend.add_trace(go.Scatter(
    x=monthly_sales["month_year"],
    y=monthly_sales["total_sales"],
    mode="lines+markers",
    name="Monthly Revenue",
    line=dict(color=C_BLUE, width=2),
    marker=dict(size=5),
    hovertemplate="<b>%{x}</b><br>Revenue: $%{y:,.0f}<extra></extra>",
))
fig_trend.add_trace(go.Scatter(
    x=monthly_sales["month_year"],
    y=monthly_sales["rolling_3m"],
    mode="lines",
    name="3-Month Rolling Avg",
    line=dict(color=C_ORANGE, width=2.5, dash="dash"),
    hovertemplate="<b>%{x}</b><br>3M Avg: $%{y:,.0f}<extra></extra>",
))
fig_trend.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="Monthly Revenue Trend (with 3-Month Rolling Avg)",
    title_font_size=15,
    xaxis_title="Month",
    yaxis_title="Revenue ($)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80, b=60, l=60, r=20),
)
fig_trend.update_xaxes(tickangle=45)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 7 — Cohort Retention Heatmap
# ─────────────────────────────────────────────────────────────────────────────
ret_vals = retention_pct.values
ret_x    = [f"Month {c}" for c in retention_pct.columns]
ret_y    = list(retention_pct.index)
text_vals = [[f"{v:.0f}%" if not np.isnan(v) else "" for v in row] for row in ret_vals]

fig_cohort = go.Figure(go.Heatmap(
    z=ret_vals,
    x=ret_x,
    y=ret_y,
    colorscale="RdYlGn",
    zmin=0,
    zmax=100,
    text=text_vals,
    texttemplate="%{text}",
    hovertemplate="<b>Cohort: %{y}</b><br>%{x}<br>Retention: %{z:.1f}%<extra></extra>",
    colorbar=dict(title="Retention %"),
))
fig_cohort.update_layout(
    template=TEMPLATE,
    height=500,
    title_text="Customer Cohort Retention Heatmap",
    title_font_size=15,
    xaxis_title="Cohort Period",
    yaxis_title="Cohort Month",
    margin=dict(t=60, b=60, l=80, r=80),
)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 8 — Target Gap Analysis
# ─────────────────────────────────────────────────────────────────────────────
tdf = monthly_target_df.copy()
bar_colors_gap = [C_GREEN if h else C_RED for h in tdf["hit"]]

fig_gap = make_subplots(specs=[[{"secondary_y": True}]])
fig_gap.add_trace(
    go.Bar(
        x=tdf["month_year"],
        y=tdf["actual_sales"],
        name="Actual Sales",
        marker_color=bar_colors_gap,
        hovertemplate="<b>%{x}</b><br>Actual: $%{y:,.0f}<extra></extra>",
    ),
    secondary_y=False,
)
fig_gap.add_trace(
    go.Scatter(
        x=tdf["month_year"],
        y=tdf["target_sales"],
        name="Target Sales",
        mode="lines+markers",
        line=dict(color=C_DARK, width=2, dash="dash"),
        marker=dict(size=5),
        hovertemplate="<b>%{x}</b><br>Target: $%{y:,.0f}<extra></extra>",
    ),
    secondary_y=False,
)
fig_gap.add_trace(
    go.Scatter(
        x=tdf["month_year"],
        y=tdf["achievement_pct"],
        name="Achievement %",
        mode="lines",
        line=dict(color=C_ORANGE, width=2),
        hovertemplate="<b>%{x}</b><br>Achievement: %{y:.1f}%<extra></extra>",
    ),
    secondary_y=True,
)
fig_gap.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="Sales Target Gap Analysis (10% YoY Growth Target)",
    title_font_size=15,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80, b=60, l=60, r=60),
)
fig_gap.update_yaxes(title_text="Sales ($)", secondary_y=False)
fig_gap.update_yaxes(title_text="Achievement %", secondary_y=True)
fig_gap.update_xaxes(tickangle=45)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 9 — RFM Segmentation (2 charts)
# ─────────────────────────────────────────────────────────────────────────────
seg_colors = {
    "Champions": C_GREEN, "Loyal": C_BLUE, "At Risk": C_ORANGE,
    "Lost": C_RED, "Potential": C_PURPLE
}
rfm_sorted = rfm_summary.sort_values("customer_count", ascending=False)

fig_rfm_count = go.Figure(go.Bar(
    x=rfm_sorted["rfm_segment"],
    y=rfm_sorted["customer_count"],
    marker_color=[seg_colors.get(s, C_GREY) for s in rfm_sorted["rfm_segment"]],
    text=rfm_sorted["customer_count"],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Customers: %{y}<extra></extra>",
))
fig_rfm_count.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="RFM Segments — Customer Count",
    title_font_size=15,
    xaxis_title="Segment",
    yaxis_title="Number of Customers",
    margin=dict(t=60, b=40, l=60, r=20),
)

fig_rfm_value = go.Figure(go.Bar(
    x=rfm_sorted["rfm_segment"],
    y=rfm_sorted["avg_monetary"].round(0),
    marker_color=[seg_colors.get(s, C_GREY) for s in rfm_sorted["rfm_segment"]],
    text=[f"${v:,.0f}" for v in rfm_sorted["avg_monetary"]],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Avg Monetary: $%{y:,.0f}<extra></extra>",
))
fig_rfm_value.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="RFM Segments — Avg Monetary Value",
    title_font_size=15,
    xaxis_title="Segment",
    yaxis_title="Avg Total Spend ($)",
    margin=dict(t=60, b=40, l=60, r=20),
)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 10 — Discount Elasticity
# ─────────────────────────────────────────────────────────────────────────────
disc_df = discount_analysis.copy()

fig_discount = make_subplots(specs=[[{"secondary_y": True}]])
fig_discount.add_trace(
    go.Bar(
        x=disc_df["discount_bin"],
        y=disc_df["order_count"],
        name="Order Count",
        marker_color=C_BLUE,
        hovertemplate="<b>%{x}</b><br>Orders: %{y:,}<extra></extra>",
    ),
    secondary_y=False,
)
fig_discount.add_trace(
    go.Scatter(
        x=disc_df["discount_bin"],
        y=disc_df["avg_margin_pct"],
        name="Avg Margin %",
        mode="lines+markers",
        line=dict(color=C_RED, width=2.5),
        marker=dict(size=8),
        hovertemplate="<b>%{x}</b><br>Avg Margin: %{y:.1f}%<extra></extra>",
    ),
    secondary_y=True,
)
fig_discount.update_layout(
    template=TEMPLATE,
    height=450,
    title_text="Discount Elasticity — How Discounts Kill Margins",
    title_font_size=15,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(t=80, b=40, l=60, r=60),
)
fig_discount.update_yaxes(title_text="Order Count", secondary_y=False)
fig_discount.update_yaxes(title_text="Avg Profit Margin (%)", secondary_y=True)

# ── Summary ───────────────────────────────────────────────────────────────────
all_figures = [fig_kpi, fig_pareto, fig_donut, fig_margin, fig_regional,
               fig_trend, fig_cohort, fig_gap, fig_rfm_count, fig_rfm_value, fig_discount]
print(f"✅ Built {len(all_figures)} Plotly charts successfully")
chart_names = [
    "Executive KPI Summary", "Pareto Analysis — 80/20 Rule (Sub-Category Revenue)",
    "Revenue Share by Category", "Profit Margin % by Category", "Regional Sales vs Profit",
    "Monthly Revenue Trend (with 3-Month Rolling Avg)", "Customer Cohort Retention Heatmap",
    "Sales Target Gap Analysis (10% YoY Growth Target)", "RFM Segments — Customer Count",
    "RFM Segments — Avg Monetary Value", "Discount Elasticity — How Discounts Kill Margins"
]
for i, name in enumerate(chart_names, 1):
    print(f"   Chart {i}: {name}")

In [ ]:
import plotly.io as pio
import os

# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_PATH      = "./superstore_dashboard.html"
DASHBOARD_TITLE  = "Superstore Sales Dashboard"

# ── Build business metrics from actual computed metrics ────────────────────────
furniture_margin  = category_summary.loc[category_summary["category"]=="Furniture",   "margin_pct"].values[0]
tech_margin       = category_summary.loc[category_summary["category"]=="Technology",  "margin_pct"].values[0]
os_margin         = category_summary.loc[category_summary["category"]=="Office Supplies","margin_pct"].values[0]
furniture_rev     = category_summary.loc[category_summary["category"]=="Furniture",   "total_sales"].values[0]
tech_rev          = category_summary.loc[category_summary["category"]=="Technology",  "total_sales"].values[0]
os_rev            = category_summary.loc[category_summary["category"]=="Office Supplies","total_sales"].values[0]

worst_region      = region_summary.sort_values("margin_pct").iloc[0]
best_reg          = region_summary.sort_values("total_profit", ascending=False).iloc[0]

n_months          = len(monthly_target_df)
months_hit        = monthly_target_df["hit"].sum()
months_missed     = n_months - months_hit
avg_achievement   = monthly_target_df["achievement_pct"].mean()

disc_0_margin     = discount_analysis.loc[discount_analysis["discount_bin"]=="0%", "avg_margin_pct"].values[0]
disc_high_margin  = discount_analysis.loc[discount_analysis["discount_bin"]=="31%+", "avg_margin_pct"].values[0]

champions_pct     = rfm_summary.loc[rfm_summary["rfm_segment"]=="Champions", "customer_count"].sum() / rfm["customer_id"].nunique() * 100
at_risk_pct       = rfm_summary.loc[rfm_summary["rfm_segment"]=="At Risk", "customer_count"].sum() / rfm["customer_id"].nunique() * 100

# ── KPI HTML cards ─────────────────────────────────────────────────────────────
kpi_html = f"""
<div class="kpi-grid">
    <div class="kpi-card">
        <div class="kpi-label">Total Revenue</div>
        <div class="kpi-value">${total_revenue:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Total Profit</div>
        <div class="kpi-value">${total_profit:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Profit Margin</div>
        <div class="kpi-value">{overall_margin_pct:.2f}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Total Orders</div>
        <div class="kpi-value">{total_orders:,}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avg Order Value</div>
        <div class="kpi-value">${avg_order_value:,.0f}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Avg Ship Days</div>
        <div class="kpi-value">{avg_ship_days:.1f} days</div>
    </div>
</div>
"""

# ── What I Found section ───────────────────────────────────────────────────────
what_i_found_html = f"""
<div style="background:white; border-radius:10px; box-shadow:0 2px 10px rgba(0,0,0,0.08); padding:25px; margin-bottom:20px;">
    <ul style="line-height:2; color:#2c3e50; font-size:0.97em; padding-left:20px;">
        <li>Furniture margin: <strong>{furniture_margin:.1f}%</strong> — discounts are too high, eroding profits.</li>
        <li>Technology is most profitable at <strong>{tech_margin:.1f}% margin</strong> but is under-promoted.</li>
        <li>No-discount orders average <strong>{disc_0_margin:.1f}%</strong> margin; 31%+ discounts drop to <strong>{disc_high_margin:.1f}%</strong>.</li>
        <li>Only <strong>{months_hit}/{n_months}</strong> months hit the 10% YoY growth target.</li>
        <li>~<strong>{at_risk_pct:.0f}%</strong> of customers are At Risk — a re-engagement campaign could recapture them.</li>
        <li>Pareto: a small group of sub-categories drives the majority of revenue.</li>
        <li>Cohort data shows significant drop-off after first purchase — acquisition stronger than retention.</li>
    </ul>
</div>
"""

# ── Business Recommendations ───────────────────────────────────────────────────
recommendations = [
    {
        "priority": "High",
        "finding": "Furniture is Losing Money",
        "evidence": f"Furniture margin is {furniture_margin:.1f}% due to heavy discounting.",
        "action": "Reduce furniture discounts and negotiate better supplier rates."
    },
    {
        "priority": "High",
        "finding": "Discounts Are Being Overused",
        "evidence": f"No-discount margin: {disc_0_margin:.1f}%. With 31%+ discounts: {disc_high_margin:.1f}%.",
        "action": "Cap discounts at 20% without manager approval."
    },
    {
        "priority": "Medium",
        "finding": "Monthly Sales Targets Are Being Missed",
        "evidence": f"Only {months_hit}/{n_months} months hit the 10% growth target. Avg achievement: {avg_achievement:.1f}%.",
        "action": "Implement mid-month check-ins to catch gaps early."
    },
    {
        "priority": "Medium",
        "finding": f"{worst_region['region']} Region Needs Attention",
        "evidence": f"Lowest margin at {worst_region['margin_pct']:.1f}% vs {best_reg['region']} at {best_reg['margin_pct']:.1f}%.",
        "action": f"Apply best practices from {best_reg['region']} to {worst_region['region']}."
    },
    {
        "priority": "Opportunity",
        "finding": "Technology Has Best Margins and is Underused",
        "evidence": f"Technology runs at {tech_margin:.1f}% margin — highest of all categories.",
        "action": "Shift marketing focus toward Technology; bundle with other categories."
    },
    {
        "priority": "Opportunity",
        "finding": "At Risk Customers Need Re-engagement",
        "evidence": f"{at_risk_pct:.0f}% of customers are At Risk; only {champions_pct:.0f}% are Champions.",
        "action": "Launch re-engagement emails or targeted offers for At Risk segment."
    },
]

rec_rows_html = ""
for r in recommendations:
    rec_rows_html += f"""
        <tr>
            <td><strong>{r['priority']}</strong></td>
            <td><strong>{r['finding']}</strong></td>
            <td style="color:#555">{r['evidence']}</td>
            <td>{r['action']}</td>
        </tr>"""

recommendations_html = f"""
<table>
    <thead>
        <tr>
            <th>Priority</th><th>Finding</th><th>Evidence</th><th>Recommended Action</th>
        </tr>
    </thead>
    <tbody>{rec_rows_html}</tbody>
</table>
"""

# ── Convert each figure to HTML div ───────────────────────────────────────────
fig_list = [
    ("pareto_div",    fig_pareto),
    ("donut_div",     fig_donut),
    ("margin_div",    fig_margin),
    ("regional_div",  fig_regional),
    ("trend_div",     fig_trend),
    ("cohort_div",    fig_cohort),
    ("gap_div",       fig_gap),
    ("rfm_count_div", fig_rfm_count),
    ("rfm_value_div", fig_rfm_value),
    ("discount_div",  fig_discount),
]

div_dict2 = {}
for idx, (key, fig) in enumerate(fig_list):
    include = (idx == 0)
    div_dict2[key] = fig.to_html(full_html=False, include_plotlyjs=include)

# ── Assemble full HTML page ───────────────────────────────────────────────────
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Superstore Sales Dashboard</title>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #f8f9fa; margin: 0; padding: 20px; }}
        h1 {{ text-align: center; color: #2c3e50; font-size: 2em; margin-bottom: 5px; }}
        .subtitle {{ text-align: center; color: #7f8c8d; margin-bottom: 30px; }}
        .section-title {{ color: #2c3e50; font-size: 1.3em; font-weight: bold; margin: 30px 0 10px 0;
                          padding-left: 10px; border-left: 4px solid #3498db; }}
        .chart-container {{ background: white; border-radius: 10px;
                            box-shadow: 0 2px 10px rgba(0,0,0,0.08); margin-bottom: 20px; padding: 15px; }}
        .chart-grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px; }}
        table {{ width: 100%; border-collapse: collapse; }}
        th {{ background: #3498db; color: white; padding: 10px; text-align: left; }}
        td {{ padding: 9px 10px; border-bottom: 1px solid #ecf0f1; vertical-align: top; font-size: 0.9em; }}
        tr:hover {{ background: #f5f9ff; }}
        .kpi-grid {{ display: grid; grid-template-columns: repeat(6, 1fr); gap: 15px; margin-bottom: 10px; }}
        .kpi-card {{ background: white; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.10);
                     padding: 20px 15px; text-align: center; border-top: 4px solid #3498db; }}
        .kpi-label {{ font-size: 0.80em; color: #7f8c8d; font-weight: 600;
                      text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px; }}
        .kpi-value {{ font-size: 1.6em; font-weight: 700; color: #2c3e50; }}
    </style>
</head>
<body>
    <h1>Superstore Sales Dashboard</h1>
    <p class="subtitle">Analysis Period: 2014&ndash;2017 &nbsp;|&nbsp; 9,994 Orders &nbsp;|&nbsp; Source: Kaggle Superstore Dataset</p>

    <div class="section-title">Executive KPI Summary</div>
    {kpi_html}

    <div class="section-title">Pareto 80/20 Analysis — Which Products Drive Revenue</div>
    <div class="chart-container">{div_dict2['pareto_div']}</div>

    <div class="section-title">Revenue & Profit Margin by Category</div>
    <div class="chart-grid-2">
        <div class="chart-container">{div_dict2['donut_div']}</div>
        <div class="chart-container">{div_dict2['margin_div']}</div>
    </div>

    <div class="section-title">Regional Performance</div>
    <div class="chart-container">{div_dict2['regional_div']}</div>

    <div class="section-title">Monthly Revenue Trend</div>
    <div class="chart-container">{div_dict2['trend_div']}</div>

    <div class="section-title">Customer Cohort Retention</div>
    <div class="chart-container">{div_dict2['cohort_div']}</div>

    <div class="section-title">Sales Target Gap Analysis</div>
    <div class="chart-container">{div_dict2['gap_div']}</div>

    <div class="section-title">RFM Customer Segmentation</div>
    <div class="chart-grid-2">
        <div class="chart-container">{div_dict2['rfm_count_div']}</div>
        <div class="chart-container">{div_dict2['rfm_value_div']}</div>
    </div>

    <div class="section-title">Discount Elasticity Analysis</div>
    <div class="chart-container">{div_dict2['discount_div']}</div>

    <div class="section-title">What I Found</div>
    {what_i_found_html}

    <div class="section-title">Business Recommendations</div>
    <div class="chart-container">{recommendations_html}</div>
</body>
</html>"""

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(html_content)

file_size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"✅ Dashboard saved to {OUTPUT_PATH} — {file_size_kb:.1f} KB")
print(f"   Charts embedded: {len(fig_list)} (KPI section uses HTML cards)")
print(f"   Recommendations: {len(recommendations)}")
print(f"   Self-contained:  Yes (Plotly JS bundled in first div)")